# Master instruction —Feature Engineering-breed

Je werkt als senior research assistant voor een masterthesis in data analysis.
Voor meer informatie over de thesis / onderzoeksvoorstel / opzet: bekijk bron "Geannoteerd_onderzoeksvoorstel.md" en voor extra gelinkte literatuur bron; "External factors and SHAP in Urban Parking copy.pdf" (beide bestanden zijn te vinden in de map 'literatuur_en_info' (binnen dit project)) 

voor structuur en gewenste flow check projectalomvattede: "README.md"

Context:
- Check hieronder de vooropgestelde feature engineering notebookstructuur:
  1. fe_00_design_and_split_lock.ipynb
  2. fe_01_build_core_features_TSE.ipynb
  3. fe_02_build_forecast_lag_features_L.ipynb
  4. fe_03_finalize_feature_sets_and_export.ipynb
- Projectfase: Phase 3 — Feature Engineering
- Phase 2 (EDA) is afgerond, dit is DE basis voor interne consistentie en keuzes voor FE!!
  - Belangrijk is ook gelijkaardige stijl als phase 2 in phase 3 over te nemen, maar dan specifiek op feature engineering toegepast natuurlijk
- shortterm is de primaire modelbasis
  - (longterm blijft enkel robustness-context en mag niet het hoofddesign sturen)
- Er zijn twee modeltracks (meer uitleg is te vinden in: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/literatuur_en_info/Geannoteerd_onderzoeksvoorstel copy.md):
  1. forecast track: autoregressieve features toegestaan
  2. policy track: occupancy-lags NIET toegestaan
- De operationele split ligt vast:
  - train = 2020, 2023, 2024
  - holdout = 2025
  - 2019 uitgesloten wegens volledig missende weatherfeatures
- Alle transformaties moeten train-only gefit worden
- Alle tijdsfeatures moeten strikt causaal opgebouwd worden
- Model- en featurekeuze gebeurt later via rolling/blocked CV binnen train
- 2025 blijft volledig locked voor finale evaluatie

Doel van phase 3:
- een reproduceerbare, leakage-safe, immutable featureset bouwen
- consistente schema’s afleveren voor forecast en policy
- features expliciet structureren in blokken (cfr. onderzoeksvoorstel):
  - T = time structure
  - S = spatial identity
  - E = external factors
  - L = autoregressive structure (alleen forecast)
- alle featurekeuzes moeten inhoudelijk verdedigbaar zijn vanuit EDA en onderzoeksdoel

Belangrijke werkinstructies:
1. Werk notebook-native en reproduceerbaar.
2. Gebruik enkel train-only fitting voor alle fit-afhankelijke transformaties.
3. Splits train en holdout vroeg en hard; nooit lekken van 2025 naar train.
4. Maak feature engineering expliciet causaal:
   - geen toekomstinformatie
   - geen target-proxies
   - geen smoothing over toekomstige observaties
   - geen niet-causale aggregaties
5. Gebruik duidelijke helperfuncties en schema-validatie.
6. Werk primair op shortterm.
7. longterm mag enkel gebruikt worden voor beperkte robustness-context, niet als primaire pipelinebasis.
8. Maak alle featurebeslissingen expliciet en academisch verdedigbaar.
9. Sluit elk notebook af met:
   - "What was built"
   - "Leakage and validity checks"
   - "Implications for next notebook"
10. Indien iets ambigu is, kies de meest leakage-veilige optie en documenteer die.

Schrijf elke interpretatieve markdown-sectie alsof ze later kan worden herwerkt tot tekst voor de masterthesis.

Stijlregels:
- helder, academisch, voorzichtig
- geen losse bullet dump als lopende tekst beter is
- benoem richting, grootteorde, onzekerheid en beperking
- maak expliciet waarom het resultaat relevant is voor de volgende fase

Jouw taak is niet alleen code schrijven, maar een thesis-waardige feature engineering pipeline ontwerpen.

## Notebookspecifieke prompt
Maak notebook `fe_00_design_and_split_lock.ipynb`.

Doel:
De feature engineering formeel ontwerpen, de temporele split hard vastzetten, de scope bepalen en een leakage-safe FE-protocol opstellen.

Context:
- Phase 2 EDA is afgerond
- shortterm is de primaire modelbasis
- longterm blijft enkel robustness-context
- tracks:
  - forecast track = met occupancy-lags
  - policy track = zonder occupancy-lags
- train = 2020, 2023, 2024
- holdout = 2025
- 2019 uitsluiten
- alle fit-afhankelijke transformaties train-only
- 2025 locked

Voer dit uit:
1. Laad de relevante Phase 1/2 data als uitgangspunt voor FE.
2. Beperk de primaire FE-pipeline tot shortterm.
3. Bevestig expliciet welke jaren in scope zijn:
   - train: 2020, 2023, 2024
   - holdout: 2025
   - exclude: 2019
4. Maak een formele feature engineering scope-definitie:
   - target
   - observation grain
   - entity grain
   - toegelaten featurefamilies
   - verboden featuretypes
5. Definieer twee formele feature tracks:
   - policy = T + S + E
   - forecast = T + S + E + L
6. Stel een leakage protocol op:
   - train-only fitting
   - time-aware lag construction
   - geen future leakage in event/cascade features
   - geen target proxies
   - geen kwaliteitsflags als predictors
   - geen hoge-cardinaliteit tekst zonder robuuste reductie
7. Maak een feature registry template met kolommen:
   - feature_name
   - block
   - source_columns
   - track_allowed
   - requires_fit
   - train_only_fit
   - causal
   - expected_role
   - notes
8. Maak een eerste ruwe lijst van candidate features op basis van EDA en onderzoeksvoorstel.
9. Definieer exportdoelen:
   - immutable train features
   - immutable holdout features
   - schema files
   - feature registry
   - data dictionary
10. Sluit af met:
   - "Final FE design decisions"
   - "Leakage guardrails"
   - "Planned build order"

Belangrijk:
- Schrijf dit notebook deels als design notebook, niet enkel als code notebook.
- Als de EDA-output in eerdere notebooks beschikbaar is, lees die en gebruik die om featurekeuzes te motiveren.
- Leg expliciet uit waarom policy en forecast apart moeten blijven.

## 0. Setup en context lock

Deze notebook formaliseert het Feature Engineering (FE)-design voor Phase 3, met expliciete focus op:

- `shortterm` als primaire modelbasis;
- een harde operationele split (`train = 2020, 2023, 2024`; `holdout = 2025`; `exclude = 2019`);
- leakage-safe ontwerpregels die identiek gelden voor forecast en policy binnen dezelfde temporele scope.

De rol van `longterm` blijft beperkt tot robustness-context en stuurt de primaire FE-pipeline niet.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data_processed").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Project root not found")


def as_flag(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce").fillna(0).gt(0)


def build_quality_mask(df: pd.DataFrame) -> pd.Series:
    mask = pd.Series(True, index=df.index)
    for col in ["system_blackout", "low_data_coverage", "partial_year", "flag_occ_inconsistent"]:
        if col in df.columns:
            mask &= ~as_flag(df[col])
    if "occupancy_rate" in df.columns:
        mask &= pd.to_numeric(df["occupancy_rate"], errors="coerce").between(0, 1, inclusive="both")
    return mask


def read_required_parquet(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    return pd.read_parquet(path)


def read_optional_csv(path: Path) -> pd.DataFrame | None:
    if not path.exists():
        return None
    return pd.read_csv(path)


def extract_points_from_markdown_heading(notebook_path: Path, heading: str) -> list[str]:
    if not notebook_path.exists():
        return []

    nb_json = json.loads(notebook_path.read_text())
    heading_norm = heading.strip().lower()

    for cell in nb_json.get("cells", []):
        if cell.get("cell_type") != "markdown":
            continue

        raw = "".join(cell.get("source", []))
        lines = [line.rstrip() for line in raw.splitlines() if line.strip()]
        if not lines:
            continue

        if lines[0].strip().lower() != heading_norm:
            continue

        points = []
        current = ""
        for line in lines[1:]:
            stripped = line.strip()
            if re.match(r"^(\d+\.|-)\s+", stripped):
                if current:
                    points.append(current.strip())
                current = re.sub(r"^(\d+\.|-)\s+", "", stripped).strip()
            else:
                if current:
                    current += " " + stripped
                else:
                    points.append(stripped)
        if current:
            points.append(current.strip())

        return [p for p in points if p]

    return []


PROJECT_ROOT = find_project_root()
DATA_INTERMEDIATE = PROJECT_ROOT / "data_intermediate"
DATA_PROCESSED = PROJECT_ROOT / "data_processed"
NB_PHASE2 = PROJECT_ROOT / "notebooks" / "phase2"

TRAIN_YEARS = [2020, 2023, 2024]
HOLDOUT_YEARS = [2025]
EXCLUDED_YEARS = [2019]

WEATHER_COLS = [
    "temp_c",
    "precip_mm",
    "wind_speed_ms",
    "wind_gusts_ms",
    "humidity_pct",
    "pressure_hpa",
    "sun_duration_min",
    "shortwave_wm2",
    "sun_intensity_wm2",
]

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2


## 1. Relevante Phase 1/2 bronnen laden als FE-uitgangspunt

We laden expliciet de kernbronnen die FE voeden:

- Phase 1 geconsolideerde inputs (`calendar`, `events`, `weather`, `parking context`);
- Phase 2 einddatasets (`MAD_shortterm`, `MAD_longterm`);
- bestaande split-artefacten (indien aanwezig) als controle op consistentie.

In [2]:
source_paths = {
    "phase1_calendar_master": DATA_INTERMEDIATE / "calendar_master.parquet",
    "phase1_events_master": DATA_INTERMEDIATE / "events_master_event_final.parquet",
    "phase1_weather_cleaned": DATA_INTERMEDIATE / "weather_cleaned.parquet",
    "phase1_parking_context": DATA_INTERMEDIATE / "parking_location_clean.parquet",
    "phase2_mad_shortterm": DATA_PROCESSED / "MAD_shortterm.parquet",
    "phase2_mad_longterm": DATA_PROCESSED / "MAD_longterm.parquet",
    "phase3_split_train": DATA_PROCESSED / "phase3_splits" / "MAD_shortterm_train_2020_2023_2024.parquet",
    "phase3_split_holdout": DATA_PROCESSED / "phase3_splits" / "MAD_shortterm_holdout_2025.parquet",
    "phase3_split_excluded": DATA_PROCESSED / "phase3_splits" / "MAD_shortterm_excluded_2019.parquet",
}

source_inventory_df = pd.DataFrame(
    [
        {
            "source": name,
            "path": str(path),
            "exists": path.exists(),
        }
        for name, path in source_paths.items()
    ]
)

display(source_inventory_df)

calendar_df = read_required_parquet(source_paths["phase1_calendar_master"])
events_df = read_required_parquet(source_paths["phase1_events_master"])
weather_df = read_required_parquet(source_paths["phase1_weather_cleaned"])
parking_ctx_df = read_required_parquet(source_paths["phase1_parking_context"])

shortterm_raw = read_required_parquet(source_paths["phase2_mad_shortterm"])
longterm_raw = read_required_parquet(source_paths["phase2_mad_longterm"])

for df in [shortterm_raw, longterm_raw]:
    df["rounded_hour"] = pd.to_datetime(df["rounded_hour"], errors="coerce")

loaded_shapes_df = pd.DataFrame(
    [
        {"dataset": "calendar_master", "rows": len(calendar_df), "cols": len(calendar_df.columns)},
        {"dataset": "events_master_event_final", "rows": len(events_df), "cols": len(events_df.columns)},
        {"dataset": "weather_cleaned", "rows": len(weather_df), "cols": len(weather_df.columns)},
        {"dataset": "parking_location_clean", "rows": len(parking_ctx_df), "cols": len(parking_ctx_df.columns)},
        {"dataset": "MAD_shortterm", "rows": len(shortterm_raw), "cols": len(shortterm_raw.columns)},
        {"dataset": "MAD_longterm", "rows": len(longterm_raw), "cols": len(longterm_raw.columns)},
    ]
)

print("Bronnen geladen voor FE design")
display(loaded_shapes_df)

,source,path,exists
0,phase1_calendar_master,/Users/emilevandevoorde/Documents/mp_mechelen_...,True
1,phase1_events_master,/Users/emilevandevoorde/Documents/mp_mechelen_...,True
2,phase1_weather_cleaned,/Users/emilevandevoorde/Documents/mp_mechelen_...,True
3,phase1_parking_context,/Users/emilevandevoorde/Documents/mp_mechelen_...,True
4,phase2_mad_shortterm,/Users/emilevandevoorde/Documents/mp_mechelen_...,True
5,phase2_mad_longterm,/Users/emilevandevoorde/Documents/mp_mechelen_...,True
6,phase3_split_train,/Users/emilevandevoorde/Documents/mp_mechelen_...,True
7,phase3_split_holdout,/Users/emilevandevoorde/Documents/mp_mechelen_...,True
8,phase3_split_excluded,/Users/emilevandevoorde/Documents/mp_mechelen_...,True


Bronnen geladen voor FE design


,dataset,rows,cols
0,calendar_master,2922,14
1,events_master_event_final,2922,15
2,weather_cleaned,52632,19
3,parking_location_clean,11,5
4,MAD_shortterm,284524,66
5,MAD_longterm,46643,65


## 2. EDA-output expliciet meenemen als motivatiebron

Om FE-keuzes traceerbaar te maken, lezen we de finale secties uit de belangrijkste Phase 2 notebooks (`eda_02` t/m `eda_07`).

In [3]:
eda_specs = [
    {
        "notebook": "eda_02_temporal_patterns.ipynb",
        "sections": ["## Key findings", "## Implications for feature engineering"],
    },
    {
        "notebook": "eda_03_spatial_patterns.ipynb",
        "sections": ["## Key findings", "## Implications for feature engineering"],
    },
    {
        "notebook": "eda_04_external_factors.ipynb",
        "sections": ["## Key findings", "## Implications for feature engineering"],
    },
    {
        "notebook": "eda_05_shortterm_vs_longterm.ipynb",
        "sections": ["## Key findings", "## Implications for feature engineering"],
    },
    {
        "notebook": "eda_06_interactions_and_hypothesis_synthesis.ipynb",
        "sections": ["## Key findings", "## Implications for feature engineering"],
    },
    {
        "notebook": "eda_07_feature_engineering_bridge.ipynb",
        "sections": [
            "## 2. Feature recommendation table (forecast track vs policy track)",
            "## 6. Operationele split voor Phase 3 (train + holdout)",
            "## Features to avoid",
        ],
    },
]

eda_rows = []
for spec in eda_specs:
    nb_path = NB_PHASE2 / spec["notebook"]
    for section in spec["sections"]:
        points = extract_points_from_markdown_heading(nb_path, section)
        if not points:
            eda_rows.append(
                {
                    "source_notebook": spec["notebook"],
                    "section": section,
                    "point_id": 1,
                    "statement": "[Geen expliciete bullet points gevonden; heading wel ingelezen.]",
                }
            )
            continue

        for i, point in enumerate(points, start=1):
            eda_rows.append(
                {
                    "source_notebook": spec["notebook"],
                    "section": section,
                    "point_id": i,
                    "statement": point,
                }
            )

eda_evidence_df = pd.DataFrame(eda_rows)
eda_evidence_compact_df = (
    eda_evidence_df.groupby(["source_notebook", "section"], as_index=False)
    .agg(n_points=("statement", "count"), first_point=("statement", "first"))
)

print("Compact EDA-evidence overzicht")
display(eda_evidence_compact_df)

print("Traceerbare EDA-evidence")
display(eda_evidence_df)

Compact EDA-evidence overzicht


,source_notebook,section,n_points,first_point
0,eda_02_temporal_patterns.ipynb,## Implications for feature engineering,4,Prioriteer cyclische tijdsfeatures en seizoens...
1,eda_02_temporal_patterns.ipynb,## Key findings,4,De data tonen duidelijke daily en weekly tempo...
2,eda_03_spatial_patterns.ipynb,## Implications for feature engineering,4,"Combineer coarse spatial features (`tier`, `lo..."
3,eda_03_spatial_patterns.ipynb,## Key findings,4,Centrum versus outer geeft een duidelijk nivea...
4,eda_04_external_factors.ipynb,## Implications for feature engineering,4,Gebruik niet-lineaire transformaties (bins/spl...
5,eda_04_external_factors.ipynb,## Key findings,4,Externe drivers zijn relevant maar sterk conte...
6,eda_05_shortterm_vs_longterm.ipynb,## Implications for feature engineering,4,Bouw de primaire feature pipeline voor shortte...
7,eda_05_shortterm_vs_longterm.ipynb,## Key findings,4,"ST en LT vormen verschillende gedragsregimes, ..."
8,eda_06_interactions_and_hypothesis_synthesis.i...,## Implications for feature engineering,5,Bouw een interaction-first feature set met gec...
9,eda_06_interactions_and_hypothesis_synthesis.i...,## Key findings,4,"Interacties zijn centraal, niet marginaal: tie..."


Traceerbare EDA-evidence


,source_notebook,section,point_id,statement
0,eda_02_temporal_patterns.ipynb,## Key findings,1,De data tonen duidelijke daily en weekly tempo...
1,eda_02_temporal_patterns.ipynb,## Key findings,2,Tijdsdynamiek verschilt tussen shortterm en lo...
2,eda_02_temporal_patterns.ipynb,## Key findings,3,Jaarverschillen zijn substantieel en beïnvloed...
3,eda_02_temporal_patterns.ipynb,## Key findings,4,Seizoensdifferentiatie maakt stationariteitste...
4,eda_02_temporal_patterns.ipynb,## Implications for feature engineering,1,Prioriteer cyclische tijdsfeatures en seizoens...
5,eda_02_temporal_patterns.ipynb,## Implications for feature engineering,2,Voeg jaar/regime-indicatoren en interacties me...
6,eda_02_temporal_patterns.ipynb,## Implications for feature engineering,3,Overweeg dataset-specifieke feature pipelines ...
7,eda_02_temporal_patterns.ipynb,## Implications for feature engineering,4,Gebruik kwaliteitsflags uit `eda_00` als sensi...
8,eda_03_spatial_patterns.ipynb,## Key findings,1,Centrum versus outer geeft een duidelijk nivea...
9,eda_03_spatial_patterns.ipynb,## Key findings,2,Capaciteit verklaart bezetting niet uniform ov...


## 3. Primaire FE-pipeline: shortterm-only + harde jaarscope

Conform EDA en onderzoeksvoorstel beperken we de primaire FE-pipeline tot `shortterm`. `longterm` blijft alleen een robustness-referentie.

In [4]:
st = shortterm_raw.copy()
lt = longterm_raw.copy()

st["quality_keep"] = build_quality_mask(st)
lt["quality_keep"] = build_quality_mask(lt)

st = st.loc[st["quality_keep"]].copy()
lt = lt.loc[lt["quality_keep"]].copy()

st["operational_split"] = np.select(
    [
        st["year"].isin(TRAIN_YEARS),
        st["year"].isin(HOLDOUT_YEARS),
    ],
    ["train", "holdout"],
    default="excluded",
)

split_year_df = (
    st.groupby(["operational_split", "year"], as_index=False)
    .size()
    .rename(columns={"size": "n_rows"})
    .sort_values(["operational_split", "year"])
)

split_overview_df = (
    st.groupby("operational_split", as_index=False)
    .agg(
        n_rows=("occupancy_rate", "size"),
        n_parkings=("parking_id", "nunique"),
        date_min=("rounded_hour", "min"),
        date_max=("rounded_hour", "max"),
    )
)
split_overview_df["pct_of_shortterm_scope"] = split_overview_df["n_rows"] / len(st) * 100

weather_missingness_by_year_df = (
    st.groupby("year")[WEATHER_COLS]
    .apply(lambda d: d.isna().mean() * 100)
    .reset_index()
)

scope_lock_df = pd.DataFrame(
    [
        {"scope_type": "train", "years": ", ".join(map(str, TRAIN_YEARS))},
        {"scope_type": "holdout", "years": ", ".join(map(str, HOLDOUT_YEARS))},
        {"scope_type": "exclude", "years": ", ".join(map(str, EXCLUDED_YEARS))},
    ]
)

# Formele checks voor split-lock
train_years_found = sorted(st.loc[st["operational_split"] == "train", "year"].dropna().unique().tolist())
holdout_years_found = sorted(st.loc[st["operational_split"] == "holdout", "year"].dropna().unique().tolist())
excluded_years_found = sorted(st.loc[st["operational_split"] == "excluded", "year"].dropna().unique().tolist())

assert train_years_found == TRAIN_YEARS, f"Train years mismatch: {train_years_found}"
assert holdout_years_found == HOLDOUT_YEARS, f"Holdout years mismatch: {holdout_years_found}"
assert EXCLUDED_YEARS[0] in excluded_years_found, f"Excluded year 2019 ontbreekt: {excluded_years_found}"

print("Scope-lock bevestigd")
display(scope_lock_df)

print("Split-overzicht")
display(split_overview_df)

print("Split per jaar")
display(split_year_df)

print("Weather-missingness per jaar (kwaliteit-gefilterde shortterm)")
display(weather_missingness_by_year_df.round(2))

primary_basis_df = pd.DataFrame(
    [
        {
            "dataset": "shortterm",
            "role": "PRIMARY_FE_PIPELINE",
            "rows_quality_filtered": int(len(st)),
            "years": ", ".join(map(str, sorted(st["year"].dropna().unique().tolist()))),
        },
        {
            "dataset": "longterm",
            "role": "ROBUSTNESS_CONTEXT_ONLY",
            "rows_quality_filtered": int(len(lt)),
            "years": ", ".join(map(str, sorted(lt["year"].dropna().unique().tolist()))),
        },
    ]
)

print("Primaire basis bevestigd")
display(primary_basis_df)

Scope-lock bevestigd


,scope_type,years
0,train,"2020, 2023, 2024"
1,holdout,2025
2,exclude,2019


Split-overzicht


,operational_split,n_rows,n_parkings,date_min,date_max,pct_of_shortterm_scope
0,excluded,33000,6,2019-01-01,2019-12-31 23:00:00,13.176967
1,holdout,87600,10,2025-01-01,2025-12-31 23:00:00,34.978857
2,train,129837,10,2020-01-01,2024-12-31 23:00:00,51.844176


Split per jaar


,operational_split,year,n_rows
0,excluded,2019,33000
1,holdout,2025,87600
2,train,2020,32741
3,train,2023,39980
4,train,2024,57116


Weather-missingness per jaar (kwaliteit-gefilterde shortterm)


,year,temp_c,precip_mm,wind_speed_ms,wind_gusts_ms,humidity_pct,pressure_hpa,sun_duration_min,shortwave_wm2,sun_intensity_wm2
0,2019,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
1,2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2023,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2024,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Primaire basis bevestigd


,dataset,role,rows_quality_filtered,years
0,shortterm,PRIMARY_FE_PIPELINE,250437,"2019, 2020, 2023, 2024, 2025"
1,longterm,ROBUSTNESS_CONTEXT_ONLY,46643,2024


De scheiding is hiermee formeel vastgezet:

- **train** bevat enkel `2020, 2023, 2024`;
- **holdout** bevat enkel `2025`;
- **2019** wordt expliciet uitgesloten uit FE-scope vanwege onverenigbaarheid met de externe weatherfamilie.

Deze lock wordt in alle volgende FE-notebooks als contract behandeld.

## 4. Formele FE-scope definitie

We definiëren de FE-scope in drie lagen:

1. kerncontract (`target`, observation grain, entity grain);
2. toegelaten featurefamilies;
3. verboden featuretypes.

In [5]:
fe_scope_core_df = pd.DataFrame(
    [
        {
            "scope_item": "target",
            "definition": "occupancy_rate",
            "operational_rule": "continue target in [0,1]; geen contemporaine target-proxy in predictors",
        },
        {
            "scope_item": "observation_grain",
            "definition": "1 rij per parking_id x rounded_hour",
            "operational_rule": "uur-resolutie behouden; geen aggregatie met toekomstige uren",
        },
        {
            "scope_item": "entity_grain",
            "definition": "parking_id als primaire entity, tier_temporal als groepscontext",
            "operational_rule": "entity encodings train-only fitten",
        },
    ]
)

allowed_feature_families_df = pd.DataFrame(
    [
        {
            "block": "T",
            "family": "time structure",
            "examples": "hour_sin/cos, weekday_sin/cos, month_sin/cos, holidays, school vacation",
            "allowed_in": "policy + forecast",
        },
        {
            "block": "S",
            "family": "spatial identity",
            "examples": "parking_id encoding, tier_temporal, capacity context",
            "allowed_in": "policy + forecast",
        },
        {
            "block": "E",
            "family": "external factors",
            "examples": "weather transforms, event flags, event cascade, interactions",
            "allowed_in": "policy + forecast",
        },
        {
            "block": "L",
            "family": "autoregressive occupancy structure",
            "examples": "occ_lag_1h/24h/168h, causal rolling windows",
            "allowed_in": "forecast only",
        },
    ]
)

forbidden_feature_types_df = pd.DataFrame(
    [
        {
            "forbidden_type": "target proxies",
            "examples": "occupancy_rate (t), number_of_occupied_spaces (t)",
            "why_forbidden": "leakage en pseudo-performance",
        },
        {
            "forbidden_type": "future-informed temporal transforms",
            "examples": "centered rolling windows, lead features",
            "why_forbidden": "future leakage",
        },
        {
            "forbidden_type": "quality flags as predictors",
            "examples": "system_blackout, low_data_coverage, partial_year",
            "why_forbidden": "kwaliteitssignalen modelleren artefacten i.p.v. gedrag",
        },
        {
            "forbidden_type": "high-cardinality free text zonder robuuste reductie",
            "examples": "event_names_combined, holiday_name strings",
            "why_forbidden": "instabiliteit, overfitting, slechte reproduceerbaarheid",
        },
    ]
)

print("FE scope core")
display(fe_scope_core_df)

print("Toegelaten featurefamilies")
display(allowed_feature_families_df)

print("Verboden featuretypes")
display(forbidden_feature_types_df)

FE scope core


,scope_item,definition,operational_rule
0,target,occupancy_rate,"continue target in [0,1]; geen contemporaine t..."
1,observation_grain,1 rij per parking_id x rounded_hour,uur-resolutie behouden; geen aggregatie met to...
2,entity_grain,"parking_id als primaire entity, tier_temporal ...",entity encodings train-only fitten


Toegelaten featurefamilies


,block,family,examples,allowed_in
0,T,time structure,"hour_sin/cos, weekday_sin/cos, month_sin/cos, ...",policy + forecast
1,S,spatial identity,"parking_id encoding, tier_temporal, capacity c...",policy + forecast
2,E,external factors,"weather transforms, event flags, event cascade...",policy + forecast
3,L,autoregressive occupancy structure,"occ_lag_1h/24h/168h, causal rolling windows",forecast only


Verboden featuretypes


,forbidden_type,examples,why_forbidden
0,target proxies,"occupancy_rate (t), number_of_occupied_spaces (t)",leakage en pseudo-performance
1,future-informed temporal transforms,"centered rolling windows, lead features",future leakage
2,quality flags as predictors,"system_blackout, low_data_coverage, partial_year",kwaliteitssignalen modelleren artefacten i.p.v...
3,high-cardinality free text zonder robuuste red...,"event_names_combined, holiday_name strings","instabiliteit, overfitting, slechte reproducee..."


## 5. Twee formele tracks: policy (T+S+E) versus forecast (T+S+E+L)

De scheiding tussen policy en forecast is methodologisch noodzakelijk:

- **Forecast track** maximaliseert kortetermijnnauwkeurigheid en mag daarom inertie (`L`) gebruiken;
- **Policy track** moet scenario-gevoelig blijven en sluit occupancy-lags uit, zodat simulaties niet vooral historische inertie reproduceren.

Beide tracks gebruiken dezelfde operationele rijen (zelfde split), maar niet dezelfde featuretoegang.

In [6]:
track_definition_df = pd.DataFrame(
    [
        {
            "track": "policy",
            "formal_formula": "T + S + E",
            "occupancy_lags_allowed": False,
            "primary_use": "policy simulation and scenario sensitivity",
            "why_separate": "voorkomt dat scenario-uitkomsten artificieel door autoregressieve inertie worden gedomineerd",
            "main_risk_if_mixed": "beleidsinterpretatie vervuild door historisch padafhankelijk gedrag",
        },
        {
            "track": "forecast",
            "formal_formula": "T + S + E + L",
            "occupancy_lags_allowed": True,
            "primary_use": "short-term predictive accuracy",
            "why_separate": "lags leveren sterke voorspelkracht volgens EDA, maar zijn niet policy-neutraal",
            "main_risk_if_mixed": "schijnbaar hoge performantie wordt foutief als policy-effect geïnterpreteerd",
        },
    ]
)

track_family_matrix_df = pd.DataFrame(
    [
        {"feature_block": "T", "policy": 1, "forecast": 1},
        {"feature_block": "S", "policy": 1, "forecast": 1},
        {"feature_block": "E", "policy": 1, "forecast": 1},
        {"feature_block": "L", "policy": 0, "forecast": 1},
    ]
)

print("Track-definitie")
display(track_definition_df)

print("Track x featureblok matrix (1=toegelaten)")
display(track_family_matrix_df)

Track-definitie


,track,formal_formula,occupancy_lags_allowed,primary_use,why_separate,main_risk_if_mixed
0,policy,T + S + E,False,policy simulation and scenario sensitivity,voorkomt dat scenario-uitkomsten artificieel d...,beleidsinterpretatie vervuild door historisch ...
1,forecast,T + S + E + L,True,short-term predictive accuracy,lags leveren sterke voorspelkracht volgens EDA...,schijnbaar hoge performantie wordt foutief als...


Track x featureblok matrix (1=toegelaten)


,feature_block,policy,forecast
0,T,1,1
1,S,1,1
2,E,1,1
3,L,0,1


## 6. Leakage-safe FE protocol

Onderstaande regels zijn bindend voor alle volgende FE-buildstappen.

In [7]:
leakage_protocol_df = pd.DataFrame(
    [
        {
            "protocol_id": "LP1",
            "rule": "Train-only fitting",
            "implementation_detail": "alle encoders/binners/splines fitten op train-jaren (2020, 2023, 2024)",
            "applies_to": "policy + forecast",
            "validation_check": "fit-statistieken worden niet berekend op holdout 2025",
        },
        {
            "protocol_id": "LP2",
            "rule": "Time-aware lag construction",
            "implementation_detail": "lags per parking_id met past-only shifts en NaN bij gaten",
            "applies_to": "forecast",
            "validation_check": "geen negatieve time delta in lag-bronrelaties",
        },
        {
            "protocol_id": "LP3",
            "rule": "Geen future leakage in event/cascade features",
            "implementation_detail": "alleen eventinformatie gebruiken die ex ante beschikbaar is op predictietijdstip",
            "applies_to": "policy + forecast",
            "validation_check": "featuredefinition documenteert availability timestamp",
        },
        {
            "protocol_id": "LP4",
            "rule": "Geen target proxies",
            "implementation_detail": "occupancy op t en afgeleiden daarvan niet gebruiken als contemporaine predictor",
            "applies_to": "policy + forecast",
            "validation_check": "feature registry markeert verboden kolommen",
        },
        {
            "protocol_id": "LP5",
            "rule": "Geen kwaliteitsflags als predictors",
            "implementation_detail": "quality flags enkel gebruiken voor filtering/monitoring, niet als modelinput",
            "applies_to": "policy + forecast",
            "validation_check": "predictor-lijsten bevatten geen quality flags",
        },
        {
            "protocol_id": "LP6",
            "rule": "Geen hoge-cardinaliteit tekst zonder robuuste reductie",
            "implementation_detail": "free-text eerst reduceren (clustering/hashing) of uitsluiten",
            "applies_to": "policy + forecast",
            "validation_check": "schema check op object/text high-cardinality velden",
        },
    ]
)

display(leakage_protocol_df)

,protocol_id,rule,implementation_detail,applies_to,validation_check
0,LP1,Train-only fitting,alle encoders/binners/splines fitten op train-...,policy + forecast,fit-statistieken worden niet berekend op holdo...
1,LP2,Time-aware lag construction,lags per parking_id met past-only shifts en Na...,forecast,geen negatieve time delta in lag-bronrelaties
2,LP3,Geen future leakage in event/cascade features,alleen eventinformatie gebruiken die ex ante b...,policy + forecast,featuredefinition documenteert availability ti...
3,LP4,Geen target proxies,occupancy op t en afgeleiden daarvan niet gebr...,policy + forecast,feature registry markeert verboden kolommen
4,LP5,Geen kwaliteitsflags als predictors,quality flags enkel gebruiken voor filtering/m...,policy + forecast,predictor-lijsten bevatten geen quality flags
5,LP6,Geen hoge-cardinaliteit tekst zonder robuuste ...,free-text eerst reduceren (clustering/hashing)...,policy + forecast,schema check op object/text high-cardinality v...


## 7. Feature registry template (contract)

Minimale registrykolommen (verplicht in latere FE-notebooks):

- `feature_name`
- `block`
- `source_columns`
- `track_allowed`
- `requires_fit`
- `train_only_fit`
- `causal`
- `expected_role`
- `notes`

In [8]:
feature_registry_columns = [
    "feature_name",
    "block",
    "source_columns",
    "track_allowed",
    "requires_fit",
    "train_only_fit",
    "causal",
    "expected_role",
    "notes",
]

feature_registry_template_df = pd.DataFrame(columns=feature_registry_columns)

# Voorbeeldrijen als invulstarter
feature_registry_seed_df = pd.DataFrame(
    [
        {
            "feature_name": "hour_sin",
            "block": "T",
            "source_columns": "hour",
            "track_allowed": "policy|forecast",
            "requires_fit": False,
            "train_only_fit": False,
            "causal": True,
            "expected_role": "capturing 24h cyclicity",
            "notes": "paired with hour_cos",
        },
        {
            "feature_name": "occupancy_rate_lag_24h",
            "block": "L",
            "source_columns": "occupancy_rate",
            "track_allowed": "forecast",
            "requires_fit": False,
            "train_only_fit": False,
            "causal": True,
            "expected_role": "autoregressive seasonality",
            "notes": "strictly past-only; no policy use",
        },
        {
            "feature_name": "event_scale_max_binned",
            "block": "E",
            "source_columns": "event_scale_max",
            "track_allowed": "policy|forecast",
            "requires_fit": True,
            "train_only_fit": True,
            "causal": True,
            "expected_role": "non-linear event intensity",
            "notes": "bin edges fit on train only",
        },
    ],
    columns=feature_registry_columns,
)

print("Lege registry template")
display(feature_registry_template_df)

print("Registry seed (voorbeeld)")
display(feature_registry_seed_df)

Lege registry template


,feature_name,block,source_columns,track_allowed,requires_fit,train_only_fit,causal,expected_role,notes


Registry seed (voorbeeld)


,feature_name,block,source_columns,track_allowed,requires_fit,train_only_fit,causal,expected_role,notes
0,hour_sin,T,hour,policy|forecast,False,False,True,capturing 24h cyclicity,paired with hour_cos
1,occupancy_rate_lag_24h,L,occupancy_rate,forecast,False,False,True,autoregressive seasonality,strictly past-only; no policy use
2,event_scale_max_binned,E,event_scale_max,policy|forecast,True,True,True,non-linear event intensity,bin edges fit on train only


## 8. Eerste ruwe candidate featurelijst (EDA + onderzoeksvoorstel)

De lijst hieronder is een startpunt voor implementatie in `fe_01` (T/S/E) en `fe_02` (L). Ze reflecteert:

- EDA-signalen over temporal/spatial/external interacties;
- het onderzoeksvoorstel met expliciete T/S/E/L-structuur;
- de track-scheiding tussen policy en forecast.

In [9]:
candidate_rows = [
    {
        "feature_name": "hour_sin",
        "block": "T",
        "source_columns": "hour",
        "track_allowed": "policy|forecast",
        "requires_fit": False,
        "train_only_fit": False,
        "causal": True,
        "expected_role": "24h periodicity",
        "notes": "EDA temporal structure",
    },
    {
        "feature_name": "hour_cos",
        "block": "T",
        "source_columns": "hour",
        "track_allowed": "policy|forecast",
        "requires_fit": False,
        "train_only_fit": False,
        "causal": True,
        "expected_role": "24h periodicity",
        "notes": "paired with hour_sin",
    },
    {
        "feature_name": "weekday_sin",
        "block": "T",
        "source_columns": "weekday_int",
        "track_allowed": "policy|forecast",
        "requires_fit": False,
        "train_only_fit": False,
        "causal": True,
        "expected_role": "weekly periodicity",
        "notes": "period=7",
    },
    {
        "feature_name": "weekday_cos",
        "block": "T",
        "source_columns": "weekday_int",
        "track_allowed": "policy|forecast",
        "requires_fit": False,
        "train_only_fit": False,
        "causal": True,
        "expected_role": "weekly periodicity",
        "notes": "period=7",
    },
    {
        "feature_name": "month_sin",
        "block": "T",
        "source_columns": "month",
        "track_allowed": "policy|forecast",
        "requires_fit": False,
        "train_only_fit": False,
        "causal": True,
        "expected_role": "annual seasonality",
        "notes": "period=12",
    },
    {
        "feature_name": "month_cos",
        "block": "T",
        "source_columns": "month",
        "track_allowed": "policy|forecast",
        "requires_fit": False,
        "train_only_fit": False,
        "causal": True,
        "expected_role": "annual seasonality",
        "notes": "period=12",
    },
    {
        "feature_name": "is_weekend",
        "block": "T",
        "source_columns": "weekday_int",
        "track_allowed": "policy|forecast",
        "requires_fit": False,
        "train_only_fit": False,
        "causal": True,
        "expected_role": "day-type shift",
        "notes": "context for interactions",
    },
    {
        "feature_name": "holiday_family_flags",
        "block": "T",
        "source_columns": "is_any_holiday,is_national_holiday,is_other_holiday",
        "track_allowed": "policy|forecast",
        "requires_fit": False,
        "train_only_fit": False,
        "causal": True,
        "expected_role": "calendar shock context",
        "notes": "hierarchical holiday signal",
    },
    {
        "feature_name": "is_school_vacation",
        "block": "T",
        "source_columns": "is_school_vacation",
        "track_allowed": "policy|forecast",
        "requires_fit": False,
        "train_only_fit": False,
        "causal": True,
        "expected_role": "vacation regime",
        "notes": "tier interaction candidate",
    },
    {
        "feature_name": "parking_id_encoded",
        "block": "S",
        "source_columns": "parking_id",
        "track_allowed": "policy|forecast",
        "requires_fit": True,
        "train_only_fit": True,
        "causal": True,
        "expected_role": "entity baseline",
        "notes": "one-hot or leakage-safe encoding",
    },
    {
        "feature_name": "tier_temporal",
        "block": "S",
        "source_columns": "parking_location_category",
        "track_allowed": "policy|forecast",
        "requires_fit": False,
        "train_only_fit": False,
        "causal": True,
        "expected_role": "centrum vs vesten/rand heterogeneity",
        "notes": "core stratification feature",
    },
    {
        "feature_name": "log_total_capacity",
        "block": "S",
        "source_columns": "total_capacity",
        "track_allowed": "policy|forecast",
        "requires_fit": False,
        "train_only_fit": False,
        "causal": True,
        "expected_role": "capacity scale context",
        "notes": "stabilize right-skew",
    },
    {
        "feature_name": "event_day_flags",
        "block": "E",
        "source_columns": "is_event_day,is_football_day,is_sport_day,is_festival_day",
        "track_allowed": "policy|forecast",
        "requires_fit": False,
        "train_only_fit": False,
        "causal": True,
        "expected_role": "event shock indicators",
        "notes": "ex ante availability required",
    },
    {
        "feature_name": "n_concurrent_events_capped",
        "block": "E",
        "source_columns": "n_concurrent_events",
        "track_allowed": "policy|forecast",
        "requires_fit": True,
        "train_only_fit": True,
        "causal": True,
        "expected_role": "event pressure intensity",
        "notes": "cap fitted on train",
    },
    {
        "feature_name": "event_scale_max_binned",
        "block": "E",
        "source_columns": "event_scale_max",
        "track_allowed": "policy|forecast",
        "requires_fit": True,
        "train_only_fit": True,
        "causal": True,
        "expected_role": "non-linear event size",
        "notes": "train-only bin edges",
    },
    {
        "feature_name": "football_kickoff_proximity",
        "block": "E",
        "source_columns": "football_kickoff_hour,rounded_hour",
        "track_allowed": "policy|forecast",
        "requires_fit": False,
        "train_only_fit": False,
        "causal": True,
        "expected_role": "pre/post event timing effect",
        "notes": "only if kickoff known beforehand",
    },
    {
        "feature_name": "temp_c_spline",
        "block": "E",
        "source_columns": "temp_c",
        "track_allowed": "policy|forecast",
        "requires_fit": True,
        "train_only_fit": True,
        "causal": True,
        "expected_role": "non-linear weather effect",
        "notes": "season interaction candidate",
    },
    {
        "feature_name": "precip_mm_bins",
        "block": "E",
        "source_columns": "precip_mm",
        "track_allowed": "policy|forecast",
        "requires_fit": True,
        "train_only_fit": True,
        "causal": True,
        "expected_role": "threshold precipitation behavior",
        "notes": "dry/light/moderate/heavy bins",
    },
    {
        "feature_name": "wind_speed_spline",
        "block": "E",
        "source_columns": "wind_speed_ms,wind_gusts_ms",
        "track_allowed": "policy|forecast",
        "requires_fit": True,
        "train_only_fit": True,
        "causal": True,
        "expected_role": "wind sensitivity",
        "notes": "include strong-wind threshold",
    },
    {
        "feature_name": "event_x_tier",
        "block": "E",
        "source_columns": "event flags x tier_temporal",
        "track_allowed": "policy|forecast",
        "requires_fit": False,
        "train_only_fit": False,
        "causal": True,
        "expected_role": "tier-specific event heterogeneity",
        "notes": "key from EDA interactions",
    },
    {
        "feature_name": "weather_x_season",
        "block": "E",
        "source_columns": "weather vars x season",
        "track_allowed": "policy|forecast",
        "requires_fit": False,
        "train_only_fit": False,
        "causal": True,
        "expected_role": "season-modulated weather effects",
        "notes": "supports non-stationary weather response",
    },
    {
        "feature_name": "event_x_hour",
        "block": "E",
        "source_columns": "is_event_day x hour_sin/hour_cos",
        "track_allowed": "policy|forecast",
        "requires_fit": False,
        "train_only_fit": False,
        "causal": True,
        "expected_role": "intra-day event timing",
        "notes": "stronger than event-day only",
    },
    {
        "feature_name": "occupancy_rate_lag_1h",
        "block": "L",
        "source_columns": "occupancy_rate",
        "track_allowed": "forecast",
        "requires_fit": False,
        "train_only_fit": False,
        "causal": True,
        "expected_role": "short memory autoregression",
        "notes": "strict shift within parking_id",
    },
    {
        "feature_name": "occupancy_rate_lag_24h",
        "block": "L",
        "source_columns": "occupancy_rate",
        "track_allowed": "forecast",
        "requires_fit": False,
        "train_only_fit": False,
        "causal": True,
        "expected_role": "daily autoregression",
        "notes": "align by hourly timestamp",
    },
    {
        "feature_name": "occupancy_rate_lag_168h",
        "block": "L",
        "source_columns": "occupancy_rate",
        "track_allowed": "forecast",
        "requires_fit": False,
        "train_only_fit": False,
        "causal": True,
        "expected_role": "weekly autoregression",
        "notes": "enforce gap-aware computation",
    },
    {
        "feature_name": "occupancy_rollmean_24h",
        "block": "L",
        "source_columns": "occupancy_rate",
        "track_allowed": "forecast",
        "requires_fit": False,
        "train_only_fit": False,
        "causal": True,
        "expected_role": "local regime smoothing",
        "notes": "left-closed past-only window",
    },
    {
        "feature_name": "occupancy_rollstd_24h",
        "block": "L",
        "source_columns": "occupancy_rate",
        "track_allowed": "forecast",
        "requires_fit": False,
        "train_only_fit": False,
        "causal": True,
        "expected_role": "local volatility",
        "notes": "no centered windows",
    },
]

candidate_features_df = pd.DataFrame(candidate_rows, columns=feature_registry_columns)

candidate_block_summary_df = (
    candidate_features_df.groupby(["block", "track_allowed"], as_index=False)
    .size()
    .rename(columns={"size": "n_features"})
    .sort_values(["block", "track_allowed"])
)

display(candidate_features_df)
print("Samenvatting candidate features")
display(candidate_block_summary_df)

,feature_name,block,source_columns,track_allowed,requires_fit,train_only_fit,causal,expected_role,notes
0,hour_sin,T,hour,policy|forecast,False,False,True,24h periodicity,EDA temporal structure
1,hour_cos,T,hour,policy|forecast,False,False,True,24h periodicity,paired with hour_sin
2,weekday_sin,T,weekday_int,policy|forecast,False,False,True,weekly periodicity,period=7
3,weekday_cos,T,weekday_int,policy|forecast,False,False,True,weekly periodicity,period=7
4,month_sin,T,month,policy|forecast,False,False,True,annual seasonality,period=12
5,month_cos,T,month,policy|forecast,False,False,True,annual seasonality,period=12
6,is_weekend,T,weekday_int,policy|forecast,False,False,True,day-type shift,context for interactions
7,holiday_family_flags,T,"is_any_holiday,is_national_holiday,is_other_ho...",policy|forecast,False,False,True,calendar shock context,hierarchical holiday signal
8,is_school_vacation,T,is_school_vacation,policy|forecast,False,False,True,vacation regime,tier interaction candidate
9,parking_id_encoded,S,parking_id,policy|forecast,True,True,True,entity baseline,one-hot or leakage-safe encoding


Samenvatting candidate features


,block,track_allowed,n_features
0,E,policy|forecast,10
1,L,forecast,5
2,S,policy|forecast,3
3,T,policy|forecast,9


## 9. Exportdoelen voor immutable FE-artifacts

De FE-build in `fe_01` t/m `fe_03` moet uitmonden in immutable, schema-gedekte outputbestanden.

In [10]:
export_targets_df = pd.DataFrame(
    [
        {
            "artifact": "train_features_policy.parquet",
            "scope": "train (2020, 2023, 2024)",
            "immutability_rule": "once exported, no row-level mutation",
            "content": "T+S+E predictors + target + keys",
        },
        {
            "artifact": "holdout_features_policy.parquet",
            "scope": "holdout (2025)",
            "immutability_rule": "never used for fitting/tuning",
            "content": "T+S+E predictors + target + keys",
        },
        {
            "artifact": "train_features_forecast.parquet",
            "scope": "train (2020, 2023, 2024)",
            "immutability_rule": "derived from same split lock as policy",
            "content": "T+S+E+L predictors + target + keys",
        },
        {
            "artifact": "holdout_features_forecast.parquet",
            "scope": "holdout (2025)",
            "immutability_rule": "never used for fitting/tuning",
            "content": "T+S+E+L predictors + target + keys",
        },
        {
            "artifact": "feature_schema_policy.json",
            "scope": "policy track",
            "immutability_rule": "schema versioned and hash-locked",
            "content": "column order, dtypes, nullability, feature blocks",
        },
        {
            "artifact": "feature_schema_forecast.json",
            "scope": "forecast track",
            "immutability_rule": "schema versioned and hash-locked",
            "content": "column order, dtypes, nullability, feature blocks",
        },
        {
            "artifact": "feature_registry.csv",
            "scope": "both tracks",
            "immutability_rule": "changes only via explicit version bump",
            "content": "registry metadata per feature",
        },
        {
            "artifact": "data_dictionary_features.csv",
            "scope": "both tracks",
            "immutability_rule": "must match schema and registry",
            "content": "human-readable definitions and units",
        },
    ]
)

display(export_targets_df)

,artifact,scope,immutability_rule,content
0,train_features_policy.parquet,"train (2020, 2023, 2024)","once exported, no row-level mutation",T+S+E predictors + target + keys
1,holdout_features_policy.parquet,holdout (2025),never used for fitting/tuning,T+S+E predictors + target + keys
2,train_features_forecast.parquet,"train (2020, 2023, 2024)",derived from same split lock as policy,T+S+E+L predictors + target + keys
3,holdout_features_forecast.parquet,holdout (2025),never used for fitting/tuning,T+S+E+L predictors + target + keys
4,feature_schema_policy.json,policy track,schema versioned and hash-locked,"column order, dtypes, nullability, feature blocks"
5,feature_schema_forecast.json,forecast track,schema versioned and hash-locked,"column order, dtypes, nullability, feature blocks"
6,feature_registry.csv,both tracks,changes only via explicit version bump,registry metadata per feature
7,data_dictionary_features.csv,both tracks,must match schema and registry,human-readable definitions and units


## Final FE design decisions

1. De primaire FE-pipeline is strikt `shortterm`-gebaseerd; `longterm` blijft robustness-context.
2. De operationele split is hard gelockt: `train = 2020, 2023, 2024`; `holdout = 2025`; `2019 = excluded`.
3. FE wordt formeel opgebouwd in blokken `T`, `S`, `E`, `L`, met track-contracten:
   - `policy = T + S + E`
   - `forecast = T + S + E + L`
4. Policy en forecast blijven gescheiden omdat hun doel verschillend is:
   - forecast optimaliseert predictienauwkeurigheid;
   - policy bewaart scenario-gevoeligheid zonder autoregressieve inertie-dominantie.
5. Alle fit-afhankelijke transformaties (bins/splines/encoders) zijn per definitie train-only.

## Leakage guardrails

1. Geen enkele transformatie of hyperparameterkeuze mag informatie uit 2025 gebruiken.
2. Lags en rolling features worden uitsluitend causaal (past-only, gap-aware) berekend.
3. Event/cascadefeatures mogen enkel op ex ante bekende informatie steunen.
4. Target-proxies en kwaliteitsflags zijn verboden als predictors.
5. Hoge-cardinaliteit tekstfeatures zijn verboden zonder expliciete, robuuste reductiestap.
6. Feature registry en schema fungeren als audittrail: elke feature moet causaliteit en fitlogica expliciet declareren.

## Planned build order

1. `fe_01_build_core_features_TSE.ipynb`
   - build + valideren van alle `T/S/E` features op train;
   - transformatiestappen fitten op train en toepassen op holdout.
2. `fe_02_build_forecast_lag_features_L.ipynb`
   - causale lag/rolling families toevoegen voor forecast;
   - extra leakage-checks op temporal offsets.
3. `fe_03_finalize_feature_sets_and_export.ipynb`
   - finale policy/forecast feature sets assembleren;
   - immutable export, schema-files, feature registry en data dictionary opleveren.

## What was built

Dit notebook levert een formeel FE-designcontract, inclusief split-lock, track-definitie, leakage-protocol, registrytemplate, candidate featurelijst en exportdoelen.

## Leakage and validity checks

De split-lock is geassert op jaarniveau; de leakage-regels zijn geformaliseerd in protocolvorm en worden in volgende notebooks als harde validatiecriteria toegepast.

## Implications for next notebook

`fe_01` kan nu direct starten met de implementatie van `T/S/E` op de gelockte split, zonder nog designkeuzes open te laten.